In [2]:
import polars as pl
import os

import pandas as pd
import caveclient

%load_ext autoreload
%autoreload 2

# MICRONS

In [2]:
datalake_root = "/data/microns1412/"

In [3]:
featdef_df = pl.read_delta(os.path.join(datalake_root, 'cellfeaturedefinition'))

In [24]:
featdef_df.head()

id,description,unit,data_type,range_min,range_max
str,str,str,str,f64,f64
"""x_umap""","""umap low dimensional embedding…","""ARBITRARY_UNIT""","""<f4""",null,null
"""y_umap""","""umap low dimensional embedding…","""ARBITRARY_UNIT""","""<f4""",null,null
"""x_medial-lateral""","""The x coordinate in minnie65 a…","""MICRONS_LENGTH""","""<f4""",null,null
"""y_dorsal-ventral""","""The y coordinate in minnie65 a…","""MICRONS_LENGTH""","""<f4""",null,null
"""z_caudal-rostral""","""The z coordinate in minnie65 a…","""MICRONS_LENGTH""","""<f4""",null,null


In [17]:
print("\n".join(featdef_df["id"].to_list()))

x_umap
y_umap
x_medial-lateral
y_dorsal-ventral
z_caudal-rostral
nucleus_volume_um
nucleus_area_um
nuclear_area_to_volume_ratio
nuclear_folding_area_um
fraction_nuclear_folding
nucleus_to_soma_ratio
soma_volume_um
soma_area_um
soma_to_nucleus_center_dist
soma_area_to_volume_ratio
soma_synapse_density_um
tip_len_dist_dendrite_p75
tip_tort_dendrite_p75
num_syn_dendrite
num_syn_soma
path_length_dendrite
radial_extent_dendrite
syn_dist_distribution_dendrite_p50
syn_size_distribution_soma_p50
syn_size_distribution_dendrite_p50
syn_size_distribution_dendrite_p5
syn_size_distribution_dendrite_p95
syn_size_dendrite_cv
syn_depth_dist_p1
syn_depth_dist_p99
syn_depth_extent
median_density
radius_dist
area_factor
dendrite_length_binned_0
dendrite_length_binned_1
dendrite_length_binned_2
dendrite_length_binned_3
syn_dist_distribution_dendrite_spine_p50
syn_dist_distribution_dendrite_shaft_p50
dend_spine_shaft_offset
syn_size_distribution_spine_dendrite_p50
syn_size_distribution_spine_dendrite_p5
sy

### Semantic grouping of MICRONS / Minnie features

Features can be grouped by meaning (useful for cross-dataset comparison with VISp, e.g. soma depth, dendrite length, clustering-related summaries):

1. **Soma / cortical position (incl. “pia depth”)**
   - `x_medial-lateral`, `y_dorsal-ventral`, `z_caudal-rostral`  
   - Standard-transform coordinates; *y* ≈ 0 at pia → **soma depth from pia**. *VISp analogue: `soma_aligned_dist_from_pia`.*

2. **Embedding / visualization (non-physical)**
   - `x_umap`, `y_umap`  
   - Low-dim embedding for plotting; not in physical units.

3. **Nucleus**
   - `nucleus_volume_um`, `nucleus_area_um`, `nuclear_area_to_volume_ratio`, `nuclear_folding_area_um`, `fraction_nuclear_folding`

4. **Soma (size, shape, relation to nucleus)**
   - `nucleus_to_soma_ratio`, `soma_volume_um`, `soma_area_um`, `soma_to_nucleus_center_dist`, `soma_area_to_volume_ratio`, `soma_synapse_density_um`

5. **Dendrite structure / length / extent** (semantically “dendrite–axon length” type; overlap with VISp morph)
   - **Total path & extent:** `path_length_dendrite`, `radial_extent_dendrite`
   - **Length in space:** `dendrite_length_binned_0` … `dendrite_length_binned_3`
   - **Tip shape:** `tip_len_dist_dendrite_p75`, `tip_tort_dendrite_p75`
   - **Arbor shape (SVD of path-length vs distance):** `branch_svd0` … `branch_svd4`  
   - *VISp analogues: `basal_dendrite_total_length`, `*_max_path_distance`, `*_extent_x/y`, `*_num_branches`, etc.*

6. **Synapse depth relative to soma** (“where” along dendrite)
   - `ego_count_pca0`, `ego_count_pca1`, `ego_count_pca2` (PCs of synapse depth vs soma)  
   - `syn_depth_dist_p1`, `syn_depth_dist_p99`, `syn_depth_extent`  
   - *Conceptually related to depth/layer; often used with cluster (E–I) type.*

7. **Synapse on dendrite (counts, spine vs shaft, size/distance distributions)**
   - Counts: `num_syn_dendrite`, `num_syn_soma`, `num_spine_syn_dendrite`, `num_shaft_syn_dendrite`, `num_spine_syn_soma`
   - Distance: `syn_dist_distribution_dendrite_p50`, `syn_dist_distribution_dendrite_spine_p50`, `syn_dist_distribution_dendrite_shaft_p50`, `dend_spine_shaft_offset`
   - Size: `syn_size_distribution_dendrite_*`, `syn_size_distribution_spine_dendrite_*`, `syn_size_distribution_shaft_dendrite_*`, `syn_size_dendrite_cv`, etc.
   - Fractions: `frac_syn_spine_soma`, `frac_syn_spine_dendrite`, `syn_spine_shaft_ratio_dendrite`
   - Binned counts: `syn_count_dist_binned_shaft_0..3`, `syn_count_dist_binned_spine_0..3`, `syn_count_dist_binned_ratio_0..3`
   - **Synapse distribution PCs:** `syn_count_pca0` … `syn_count_pca9` (summaries of synapse distribution; used for clustering)

8. **Synapse on soma**
   - `syn_size_distribution_soma_p50` (plus spine/shaft counts above)

9. **Density / spatial (abstract)**
   - `median_density`, `median_density_spine`, `median_density_shaft`, `radius_dist`, `area_factor`

**Note:** E–I / cell type (“cluster”) labels (neuron, glutamatergic, gabaergic, L4IT, PTC, …) come from the **cluster** and **clustermembership** tables, not from the feature list. The *csm_cluster_features* set (groups 3–9) is what’s used for clustering and aligns conceptually with “cluster-related” features.

### soma position info

In [4]:
anatomy = pl.read_delta("/data/microns1412/cellfeatures/minnie65_std_transform_coordinates").drop(['project_id','feature_set_id'])
anatomy.head()

id,x_medial-lateral,y_dorsal-ventral,z_caudal-rostral
str,f32,f32,f32
"""373879""",828.189697,638.553772,783.719971
"""201858""",510.690918,505.672302,1050.680054
"""600774""",1255.059204,821.799255,777.679993
"""408486""",891.15741,662.693665,1002.960022
"""598774""",1235.960083,809.527954,828.52002


### E-I type info (cluster)

In [15]:
clusters=pl.read_delta("/data/microns1412/cluster")
clusters.head()

# Print all cluster classes by level (level 0 = major_class, 1 = class, 2 = subtype)
print("MICRONS / Minnie cluster classes by level:")
for lev in sorted(clusters["level"].unique().to_list()):
    cat = clusters.filter(pl.col("level") == lev)["heirachy_category"].unique().to_list()[0]
    ids = sorted(clusters.filter(pl.col("level") == lev)["id"].to_list())
    print(f"  Level {lev} ({cat}):", ids)
print("\nMICRONS / Minnie cluster classes (all, sorted):")
for c in sorted(clusters["id"].unique().to_list()):
    print(" ", c)

id,parent,children,level,score,hex_color,heirachy_category,distance_to_parent,project_id
str,str,list[str],i64,f64,str,str,f64,str
"""neuron""",null,"[""glutamatergic"", ""gabaergic""]",0,null,"""#000000""","""major_class""",null,"""minnie65"""
"""gabaergic""","""neuron""","[""PTC"", ""DTC"", … ""STC""]",1,null,"""#0000FF""","""class""",null,"""minnie65"""
"""glutamatergic""","""neuron""","[""L4IT"", ""L6CT"", … ""L6SP""]",1,null,"""#FF0000""","""class""",null,"""minnie65"""
"""PTC""","""gabaergic""",null,2,null,"""#364a7a""","""subtype""",null,"""minnie65"""
"""DTC""","""gabaergic""",null,2,null,"""#80c5c0""","""subtype""",null,"""minnie65"""


# Minnie

In [19]:
client = caveclient.CAVEclient("minnie65_phase3_v1", auth_token=os.environ["CUSTOM_KEY"])
version = 1412
client.materialize.version = version

In [27]:
featdef_df_minnie = pd.read_csv("../data/minnie1412/minnie_cell_features.csv")

In [28]:
featdef_df_minnie.head()

,id,description,unit,data_type,range_min,range_max
0,nucleus_volume_um,Nucleus volume,MICRONS_CUBED,<f4,0.0,NaN
1,nucleus_area_um,Nucleus surface area,MICRONS_SQUARE,<f4,0.0,NaN
2,nuclear_area_to_volume_ratio,Nucleus surface area to volume ratio,MICRONS_INVERSE,<f4,0.0,NaN
3,nuclear_folding_area_um,Area of nucleus in an infolding (see Elabbady ...,MICRONS_SQUARE,<f4,0.0,NaN
4,fraction_nuclear_folding,Fraction of nucleus in an infolding,RATIO,<f4,0.0,1.0


In [29]:
print("\n".join(featdef_df_minnie["id"].to_list()))

nucleus_volume_um
nucleus_area_um
nuclear_area_to_volume_ratio
nuclear_folding_area_um
fraction_nuclear_folding
nucleus_to_soma_ratio
soma_volume_um
soma_area_um
soma_to_nucleus_center_dist
soma_area_to_volume_ratio
soma_synapse_density_um
tip_len_dist_dendrite_p75
tip_tort_dendrite_p75
num_syn_dendrite
num_syn_soma
path_length_dendrite
radial_extent_dendrite
syn_dist_distribution_dendrite_p50
syn_size_distribution_soma_p50
syn_size_distribution_dendrite_p50
syn_size_distribution_dendrite_p5
syn_size_distribution_dendrite_p95
syn_size_dendrite_cv
syn_depth_dist_p1
syn_depth_dist_p99
syn_depth_extent
median_density
radius_dist
area_factor
dendrite_length_binned_0
dendrite_length_binned_1
dendrite_length_binned_2
dendrite_length_binned_3
syn_dist_distribution_dendrite_spine_p50
syn_dist_distribution_dendrite_shaft_p50
dend_spine_shaft_offset
syn_size_distribution_spine_dendrite_p50
syn_size_distribution_spine_dendrite_p5
syn_size_distribution_spine_dendrite_p95
syn_size_spine_dendrite_cv

# VISP

In [32]:
visp_exc_featset_df = pd.read_csv("../data/visp-features-and-mapping/exc_visp_patchseq_morph_feature_definitions.csv")

In [33]:
visp_inh_featset_df = pd.read_csv("../data/visp-features-and-mapping/inh_visp_patchseq_morph_feature_definitions.csv")

In [9]:
# WNM morph features
wnm_feature_matrix = pd.read_csv("/data/visp-features-and-mapping/RawFeaturesWide_ChamferCorr.csv", index_col=0)

In [40]:
print("\n".join(visp_exc_featset_df["id"].to_list()))

apical_dendrite_bias_x
apical_dendrite_bias_y
apical_dendrite_depth_pc_0
apical_dendrite_depth_pc_1
apical_dendrite_depth_pc_2
apical_dendrite_depth_pc_3
apical_dendrite_early_branch_path
apical_dendrite_emd_with_basal_dendrite
apical_dendrite_extent_x
apical_dendrite_extent_y
apical_dendrite_frac_above_basal_dendrite
apical_dendrite_frac_below_basal_dendrite
apical_dendrite_frac_intersect_basal_dendrite
apical_dendrite_max_branch_order
apical_dendrite_max_euclidean_distance
apical_dendrite_max_path_distance
apical_dendrite_mean_contraction
apical_dendrite_mean_diameter
apical_dendrite_mean_moments_along_max_distance_projection
apical_dendrite_num_branches
apical_dendrite_num_outer_bifurcations
apical_dendrite_soma_percentile_x
apical_dendrite_soma_percentile_y
apical_dendrite_std_moments_along_max_distance_projection
apical_dendrite_total_length
apical_dendrite_total_surface_area
axon_exit_distance
axon_exit_theta
basal_dendrite_bias_x
basal_dendrite_bias_y
basal_dendrite_calculate_nu

In [41]:
print("\n".join(visp_inh_featset_df["id"].to_list()))

axon_bias_x
axon_bias_y
axon_depth_pc_0
axon_depth_pc_1
axon_depth_pc_2
axon_depth_pc_3
axon_depth_pc_4
axon_depth_pc_5
axon_emd_with_basal_dendrite
axon_exit_distance
axon_exit_theta
axon_extent_x
axon_extent_y
axon_frac_above_basal_dendrite
axon_frac_below_basal_dendrite
axon_frac_intersect_basal_dendrite
axon_max_branch_order
axon_max_euclidean_distance
axon_max_path_distance
axon_mean_contraction
axon_num_branches
axon_soma_percentile_x
axon_soma_percentile_y
axon_total_length
basal_dendrite_bias_x
basal_dendrite_bias_y
basal_dendrite_calculate_number_of_stems
basal_dendrite_extent_x
basal_dendrite_extent_y
basal_dendrite_frac_above_axon
basal_dendrite_frac_below_axon
basal_dendrite_frac_intersect_axon
basal_dendrite_max_branch_order
basal_dendrite_max_euclidean_distance
basal_dendrite_max_path_distance
basal_dendrite_mean_contraction
basal_dendrite_mean_diameter
basal_dendrite_num_branches
basal_dendrite_soma_percentile_x
basal_dendrite_soma_percentile_y
basal_dendrite_stem_exit_d

In [10]:
print("\n".join(wnm_feature_matrix.columns.to_list()))

swc_path
soma_aligned_dist_from_pia
basal_dendrite_max_euclidean_distance
apical_dendrite_num_branches
basal_dendrite_stem_exit_down
apical_dendrite_bias_y
apical_dendrite_extent_y
apical_dendrite_depth_pc_0
apical_dendrite_early_branch_path
apical_dendrite_mean_contraction
apical_dendrite_soma_percentile_y
basal_dendrite_frac_above_apical_dendrite
basal_dendrite_extent_y
apical_dendrite_max_branch_order
apical_dendrite_total_length
apical_dendrite_mean_moments_along_max_distance_projection
basal_dendrite_total_length
basal_dendrite_bias_y
apical_dendrite_depth_pc_2
apical_dendrite_soma_percentile_x
apical_dendrite_depth_pc_3
basal_dendrite_extent_x
basal_dendrite_max_branch_order
basal_dendrite_max_path_distance
apical_dendrite_frac_below_basal_dendrite
apical_dendrite_frac_above_basal_dendrite
apical_dendrite_frac_intersect_basal_dendrite
basal_dendrite_soma_percentile_y
basal_dendrite_soma_percentile_x
apical_dendrite_max_euclidean_distance
basal_dendrite_stem_exit_up
apical_dendrit

## Microns/minnie and visp cell type cluster names

### VISp cluster classes (Tasic 2018 + MET types)

Load taxonomy from data files and print all class / subclass / cluster and MET type names.

In [17]:
import json

# VISp: Tasic 2018 taxonomy (class, subclass, cluster / T-type) from anno.feather
visp_anno = pd.read_feather("../data/visp-patchseq-taxonomy-info/anno.feather")
print("VISp (Tasic 2018) — class (level 1, excitatory / inhibitory):")
for c in sorted(visp_anno["class_label"].unique()):
    print(" ", c)
print("\nVISp (Tasic 2018) — subclass (level 2):")
for c in sorted(visp_anno["subclass_label"].unique()):
    print(" ", c)
print("\nVISp (Tasic 2018) — cluster / T-type (level 3):")
for c in sorted(visp_anno["cluster_label"].unique()):
    print(" ", c)

# VISp: MET types (inhibitory) from met_type_colors.json
with open("../data/visp-patchseq-taxonomy-info/met_type_colors.json", "r") as f:
    met_colors = json.load(f)
print("\nVISp MET types (inhibitory):")
for name in sorted(met_colors.keys()):
    print(" ", name)

VISp (Tasic 2018) — class (level 1, excitatory / inhibitory):
  GABAergic
  Glutamatergic
  Non-Neuronal

VISp (Tasic 2018) — subclass (level 2):
  Astro
  CR
  Endo
  L2/3 IT
  L4
  L5 IT
  L5 PT
  L6 CT
  L6 IT
  L6b
  Lamp5
  Macrophage
  Meis2
  NP
  Oligo
  Peri
  Pvalb
  SMC
  Serpinf1
  Sncg
  Sst
  VLMC
  Vip

VISp (Tasic 2018) — cluster / T-type (level 3):
  Astro Aqp4
  CR Lhx5
  Endo Ctla2a
  Endo Cytl1
  L2/3 IT VISp Adamts2
  L2/3 IT VISp Agmat
  L2/3 IT VISp Rrad
  L4 IT VISp Rspo1
  L5 IT VISp Batf3
  L5 IT VISp Col27a1
  L5 IT VISp Col6a1 Fezf2
  L5 IT VISp Hsd11b1 Endou
  L5 IT VISp Whrn Tox2
  L5 NP VISp Trhr Cpne7
  L5 NP VISp Trhr Met
  L5 PT VISp C1ql2 Cdh13
  L5 PT VISp C1ql2 Ptgfr
  L5 PT VISp Chrna6
  L5 PT VISp Krt80
  L5 PT VISp Lgr5
  L6 CT Nxph2 Sla
  L6 CT VISp Ctxn3 Brinp3
  L6 CT VISp Ctxn3 Sla
  L6 CT VISp Gpr139
  L6 CT VISp Krt80 Sla
  L6 CT VISp Nxph2 Wls
  L6 IT VISp Car3
  L6 IT VISp Col18a1
  L6 IT VISp Col23a1 Adamts2
  L6 IT VISp Penk Col27a1
  L

### MICRONS/Minnie

In [18]:
clusters=pl.read_delta("/data/microns1412/cluster")
clusters.head()

# Print all cluster classes by level (level 0 = major_class, 1 = class, 2 = subtype)
print("MICRONS / Minnie cluster classes by level:")
for lev in sorted(clusters["level"].unique().to_list()):
    cat = clusters.filter(pl.col("level") == lev)["heirachy_category"].unique().to_list()[0]
    ids = sorted(clusters.filter(pl.col("level") == lev)["id"].to_list())
    print(f"  Level {lev} ({cat}):", ids)
print("\nMICRONS / Minnie cluster classes (all, sorted):")
for c in sorted(clusters["id"].unique().to_list()):
    print(" ", c)

MICRONS / Minnie cluster classes by level:
  Level 0 (major_class): ['neuron']
  Level 1 (class): ['gabaergic', 'glutamatergic']
  Level 2 (subtype): ['DTC', 'ITC', 'L2IT', 'L3IT', 'L4IT', 'L5ET', 'L5IT', 'L5NP', 'L6CT', 'L6IT', 'L6SP', 'PTC', 'STC']

MICRONS / Minnie cluster classes (all, sorted):
  DTC
  ITC
  L2IT
  L3IT
  L4IT
  L5ET
  L5IT
  L5NP
  L6CT
  L6IT
  L6SP
  PTC
  STC
  gabaergic
  glutamatergic
  neuron
